# A8 – Adaptive & Corrective RAG

- **Experiment ID:** A8_ADAPTIVE
- **Adapted from:** adaptive_retrieval.ipynb, crag.ipynb
- **Purpose:** Query routing (identifier→BM25, semantic→dense, multi-hop→decompose) + relevance evaluator + one corrective retrieval round if evidence insufficient.
- **Covers Ch3:** 3.7.1 Query routing, 3.7.2 Adaptive retrieval, 3.7.4 Corrective RAG

In [ ]:
import sys, json, re
import numpy as np, pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)
EXPERIMENT_ID = 'A8_ADAPTIVE'
NOTEBOOK = '09_adaptive_crag.ipynb'
SEED = config['seed']
CONFIG_HASH = config['_config_hash']
FINAL_TOP_K = config['retrieval']['final_top_k']
CANDIDATE_TOP_K = config['retrieval']['candidate_top_k']
RRF_K = config['retrieval']['rrf_k']
MAX_RETRY = 1  # max corrective rounds

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f'LLM OK: {llm.invoke("hi").content[:30]}')

## Build Indexes

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config['paths']['raw_data']
documents = []
for pdf_file in raw_data_path.glob('*.pdf'):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
for c in chunks: c.page_content = c.page_content.replace('\t', ' ')

vectorstore = FAISS.from_documents(chunks, embeddings)
chunk_texts = [c.page_content for c in chunks]
bm25 = BM25Okapi([t.lower().split() for t in chunk_texts])

with open(PROJECT_ROOT / config['paths']['questions'], 'r') as f:
    eval_questions = json.load(f)
print(f'Ready: {len(chunks)} chunks, {len(eval_questions)} questions')

## Query Router (3.7.1)

Classifies query type and selects retrieval strategy.

In [ ]:
ROUTE_PROMPT = PromptTemplate(
    input_variables=['question'],
    template="""Classify this question into one category. Respond with ONLY the category name.
Categories:
- IDENTIFIER: contains specific codes, names, acronyms needing exact match
- SEMANTIC: natural language requiring meaning-based search
- MULTI_HOP: requires combining info from multiple sources
- OUT_OF_SCOPE: clearly unrelated to the knowledge base topic

Question: {question}
Category:"""
)
route_chain = ROUTE_PROMPT | llm

def route_query(question):
    resp = route_chain.invoke({'question': question}).content.strip().upper()
    for cat in ['IDENTIFIER', 'SEMANTIC', 'MULTI_HOP', 'OUT_OF_SCOPE']:
        if cat in resp: return cat
    return 'SEMANTIC'

## Adaptive Retrieval (3.7.2)

In [ ]:
def retrieve_by_route(question, route, k=FINAL_TOP_K):
    if route == 'OUT_OF_SCOPE':
        return []  # no retrieval
    elif route == 'IDENTIFIER':
        # BM25 priority
        scores = bm25.get_scores(question.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        return [chunks[i] for i in top_idx]
    elif route == 'MULTI_HOP':
        # Larger candidate pool
        dense = vectorstore.similarity_search(question, k=CANDIDATE_TOP_K)
        bm25_scores = bm25.get_scores(question.lower().split())
        bm25_top = [chunks[i] for i in np.argsort(bm25_scores)[::-1][:CANDIDATE_TOP_K]]
        # RRF
        rrf = {}
        for r, d in enumerate(dense, 1):
            rrf[d.page_content[:80]] = {'s': 1/(RRF_K+r), 'd': d}
        for r, d in enumerate(bm25_top, 1):
            key = d.page_content[:80]
            if key in rrf: rrf[key]['s'] += 1/(RRF_K+r)
            else: rrf[key] = {'s': 1/(RRF_K+r), 'd': d}
        sorted_r = sorted(rrf.values(), key=lambda x: x['s'], reverse=True)
        return [item['d'] for item in sorted_r[:k*2]]  # more docs for multi-hop
    else:  # SEMANTIC - hybrid
        dense = vectorstore.similarity_search(question, k=k)
        return dense

## Relevance Evaluator + Corrective Loop (3.7.4)

In [ ]:
EVAL_PROMPT = PromptTemplate(
    input_variables=['question', 'context'],
    template="""Does this context contain sufficient information to answer the question?
Respond ONLY: SUFFICIENT or INSUFFICIENT

Question: {question}
Context (first 500 chars): {context}
Verdict:"""
)
eval_chain = EVAL_PROMPT | llm

def evaluate_sufficiency(question, docs):
    if not docs: return 'INSUFFICIENT'
    ctx = ' '.join([d.page_content[:200] for d in docs[:3]])
    resp = eval_chain.invoke({'question': question, 'context': ctx[:500]}).content.strip().upper()
    return 'SUFFICIENT' if 'SUFFICIENT' in resp else 'INSUFFICIENT'

## Grounded Generation

In [ ]:
GROUNDED_PROMPT = PromptTemplate(
    input_variables=['sources', 'question'],
    template="""Answer ONLY from the sources. Cite as [S1], [S2]. If insufficient evidence, say 'Insufficient evidence.'

Sources:
{sources}

Question: {question}
Answer:"""
)
gen_chain = GROUNDED_PROMPT | llm

def format_sources(docs):
    return '\n\n'.join([f'[S{i+1}] {d.page_content}' for i, d in enumerate(docs)])

## Run Full Adaptive Pipeline

In [ ]:
all_results = []

for q in eval_questions:
    question = q['question']
    relevant_docs = q.get('relevant_documents', [])
    should_abstain = q.get('should_abstain', False)

    # Route
    with Timer() as t_route:
        route = route_query(question)

    # Retrieve
    with Timer() as t_ret:
        docs = retrieve_by_route(question, route)

    # Evaluate + corrective loop
    retrieval_rounds = 1
    with Timer() as t_eval:
        if docs and route != 'OUT_OF_SCOPE':
            verdict = evaluate_sufficiency(question, docs)
            if verdict == 'INSUFFICIENT' and retrieval_rounds <= MAX_RETRY:
                # Corrective: try broader retrieval
                docs = retrieve_by_route(question, 'MULTI_HOP')
                retrieval_rounds += 1

    # Generate
    if route == 'OUT_OF_SCOPE' or not docs:
        answer = 'Insufficient evidence in available sources.'
        predicted_abstain = True
    else:
        sources_text = format_sources(docs[:FINAL_TOP_K])
        with Timer() as t_gen:
            response = gen_chain.invoke({'sources': sources_text, 'question': question})
        answer = response.content
        predicted_abstain = 'insufficient evidence' in answer.lower()

    retrieved_ids = [d.metadata.get('source', f'c{i}') for i, d in enumerate(docs[:FINAL_TOP_K])]
    metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=FINAL_TOP_K)
    metrics['correct_abstain'] = (predicted_abstain == should_abstain)
    metrics['route'] = route
    metrics['retrieval_rounds'] = retrieval_rounds

    total_time = t_route.elapsed + t_ret.elapsed + t_eval.elapsed + (t_gen.elapsed if 't_gen' in dir() and not predicted_abstain else 0)
    all_results.append(build_result_record(
        experiment_id=EXPERIMENT_ID, notebook=NOTEBOOK,
        config_hash=CONFIG_HASH, seed=SEED,
        question_id=q['question_id'], question=question,
        answer=answer, predicted_abstain=predicted_abstain,
        metrics=metrics,
        latency={'route_seconds': t_route.elapsed, 'retrieval_seconds': t_ret.elapsed,
                 'total_seconds': total_time},
        usage={'llm_calls': 2 + (1 if not predicted_abstain else 0), 'retrieval_rounds': retrieval_rounds},
    ))
    print(f'  [{q["question_id"]}] route={route} rounds={retrieval_rounds} abstain={predicted_abstain}')

print(f'\nDone: {len(all_results)} records')

## Summary

In [ ]:
# Route distribution
routes = [r['metrics']['route'] for r in all_results]
for rt in set(routes):
    print(f'  {rt}: {routes.count(rt)}')

abstain_acc = np.mean([r['metrics']['correct_abstain'] for r in all_results])
avg_rounds = np.mean([r['usage']['retrieval_rounds'] for r in all_results])
print(f'\n  Abstention accuracy: {abstain_acc:.3f}')
print(f'  Avg retrieval rounds: {avg_rounds:.2f}')

In [ ]:
output_dir = PROJECT_ROOT / config['paths']['results']
save_jsonl(all_results, output_dir / 'A8_adaptive.jsonl')
save_config_snapshot(config, output_dir)
print('Saved.')